# Discovery with independent calibration

This study compares discovery that reserves calibration rows before selection with the earlier workflow that selected on every development response. Calibration rows are held-out observations whose prediction errors set the band width. Independent calibration changes which rows may choose the family, predictor transforms, formula, spread and stopping iteration. The prediction band uses the selected fit directly; its calibration responses cannot change that fit.

The tables keep supplied-specification and discovery tracks separate. Five fixed, untuned competitor controls receive the same selection/calibration partition as the new workflow. The earlier adaptive workflow uses its original internal split and is a whole-procedure baseline. Width leadership is a retrospective choice from eligible competitor test results, not a prespecified winner. The [precommitted protocol](discovery_calibration/PROTOCOL.md) records the rules and independent draws.

In [1]:
import gzip
import json
from pathlib import Path
import numpy as np
import polars as pl
from great_tables import GT
from IPython.display import display

folder = Path("discovery_calibration")
def read_run(phase):
    path = folder / f"{phase}.json.gz"
    return json.loads(gzip.decompress(path.read_bytes())) if path.exists() else json.loads(
        (folder / f"{phase}.json").read_text())
baseline, fixed = read_run("baseline"), read_run("fixed")
assert len(baseline["results"]) == 400 and len(fixed["results"]) == 2800
records = baseline["results"] + fixed["results"]
panels = ["absolute", "constant", "null", "sharp", "small", "correlated", "poisson", "heavy_tail"]
peers = ["catboost_split", "catboost_cqr", "lightgbm_split", "lightgbm_cqr", "sklearn_cqr"]
def mean_se(values):
    values = np.asarray(values, dtype=float)
    return float(values.mean()), float(values.std(ddof=1)/np.sqrt(len(values)))


In [2]:
summary = []
for panel in panels:
    for phase, source in [("baseline", baseline), ("fixed", fixed)]:
        for method in dict.fromkeys(row["method"] for row in source["results"]):
            rows = [r for r in source["results"] if r["panel"] == panel and r["method"] == method]
            ok = [r for r in rows if r["status"] == "ok"]
            n_data = rows[0]["n_development"]
            label = f"discovery/{phase}" if method == "discovery" else method
            row = {"panel": panel, "method": label, "complete": len(ok), "failed": len(rows)-len(ok),
                   "selection_n": n_data if phase == "baseline" else n_data//2, "band_fit_n": n_data//2}
            for metric in ("coverage", "width", "interval_score", "point_mse", "coverage_low", "coverage_high", "elapsed"):
                row[metric], row[metric+"_se"] = mean_se([r[metric] for r in ok]) if ok else (None, None)
            row["empty_sets"] = sum(r["empty_sets"] for r in ok)
            summary.append(row)
leaders = {}
for panel in panels:
    extra = 1/41 if panel == "small" else 0
    eligible = [r for r in summary if r["panel"] == panel and r["method"] in peers
                and r["complete"] == 50 and .88 <= r["coverage"] <= .92+extra]
    leaders[panel] = min(eligible, key=lambda row: row["width"])["method"] if eligible else None
summary_frame = pl.DataFrame(summary)


In [3]:
headline_rows = []
for row in summary:
    shown = {key: row[key] for key in ("panel", "method", "selection_n", "band_fit_n", "complete", "failed")}
    for metric in ("coverage", "width", "interval_score"):
        shown[metric] = "unavailable" if row[metric] is None else f"{row[metric]:.3f} ± {row[metric+'_se']:.3f}"
    headline_rows.append(shown)
headline = pl.DataFrame(headline_rows)
table = GT(headline, groupname_col="panel", rowname_col="method").tab_header(
    title="Prediction intervals by information track",
    subtitle="Means ± Monte Carlo SE; coverage target 0.90; lower response-unit width/score is better")
for index, row in enumerate(headline_rows):
    if row["method"] == leaders[row["panel"]]:
        table = table.fmt(lambda value: f"<strong>{value}</strong>", columns="width", rows=[index])
display(table.tab_spanner("Rows per draw", ["selection_n", "band_fit_n"])
    .tab_source_note("Known uses supplied structure; discovery receives data/response only. Controls are shared by both tracks. "
        "Bold marks the retrospective lowest-width complete competitor within the frozen coverage tolerance. SE is the between-draw SD divided by sqrt(50), not the uncertainty of one forecast. 50 independent draws per row; incomplete rows are descriptive and ineligible. "
        "Baseline selection reads all development responses before its later band refit; the new workflow reserves calibration first. No across-track overall winner is implied."))


GT(_tbl_data=shape: (64, 9)
┌────────────┬────────────┬───────────┬───────────┬───┬────────┬───────────┬───────────┬───────────┐
│ panel      ┆ method     ┆ selection ┆ band_fit_ ┆ … ┆ failed ┆ coverage  ┆ width     ┆ interval_ │
│ ---        ┆ ---        ┆ _n        ┆ n         ┆   ┆ ---    ┆ ---       ┆ ---       ┆ score     │
│ str        ┆ str        ┆ ---       ┆ ---       ┆   ┆ i64    ┆ str       ┆ str       ┆ ---       │
│            ┆            ┆ i64       ┆ i64       ┆   ┆        ┆           ┆           ┆ str       │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪════════╪═══════════╪═══════════╪═══════════╡
│ absolute   ┆ discovery/ ┆ 1000      ┆ 500       ┆ … ┆ 0      ┆ 0.900 ±   ┆ 3.699 ±   ┆ 4.663 ±   │
│            ┆ baseline   ┆           ┆           ┆   ┆        ┆ 0.002     ┆ 0.023     ┆ 0.018     │
│ absolute   ┆ discovery/ ┆ 500       ┆ 500       ┆ … ┆ 0      ┆ 0.899 ±   ┆ 3.565 ±   ┆ 4.500 ±   │
│            ┆ fixed      ┆           ┆           ┆   ┆        ┆ 0.002     ┆ 0.018     ┆ 0.023     │
│ absolute   ┆ known      ┆ 500       ┆ 500       ┆ … ┆ 0      ┆ 0.902 ±   ┆ 3.710 ±   ┆ 4.648 ±   │
│            ┆            ┆           ┆           ┆   ┆        ┆ 0.002     ┆ 0.019     ┆ 0.018     │
│ absolute   ┆ catboost_s ┆ 500       ┆ 500       ┆ … ┆ 0      ┆ 0.904 ±   ┆ 4.172 ±   ┆ 5.702 ±   │
│            ┆ plit       ┆           ┆           ┆   ┆        ┆ 0.002     ┆ 0.031     ┆ 0.042     │
│ absolute   ┆ catboost_c ┆ 500       ┆ 500       ┆ … ┆ 0      ┆ 0.903 ±   ┆ 3.972 ±   ┆ 5.488 ±   │
│            ┆ qr         ┆           ┆           ┆   ┆        ┆ 0.003     ┆ 0.036     ┆ 0.040     │
│ …          ┆ …          ┆ …         ┆ …         ┆ … ┆ …      ┆ …         ┆ …         ┆ …         │
│ heavy_tail ┆ catboost_s ┆ 500       ┆ 500       ┆ … ┆ 0      ┆ 0.902 ±   ┆ 3.011 ±   ┆ 4.763 ±   │
│            ┆ plit       ┆           ┆           ┆   ┆        ┆ 0.002     ┆ 0.029     ┆ 0.045     │
│ heavy_tail ┆ catboost_c ┆ 500       ┆ 500       ┆ … ┆ 0      ┆ 0.903 ±   ┆ 3.418 ±   ┆ 5.131 ±   │
│            ┆ qr         ┆           ┆           ┆   ┆        ┆ 0.002     ┆ 0.035     ┆ 0.046     │
│ heavy_tail ┆ lightgbm_s ┆ 500       ┆ 500       ┆ … ┆ 0      ┆ 0.902 ±   ┆ 3.105 ±   ┆ 4.804 ±   │
│            ┆ plit       ┆           ┆           ┆   ┆        ┆ 0.002     ┆ 0.026     ┆ 0.044     │
│ heavy_tail ┆ lightgbm_c ┆ 500       ┆ 500       ┆ … ┆ 0      ┆ 0.904 ±   ┆ 3.205 ±   ┆ 4.853 ±   │
│            ┆ qr         ┆           ┆           ┆   ┆        ┆ 0.002     ┆ 0.030     ┆ 0.044     │
│ heavy_tail ┆ sklearn_cq ┆ 500       ┆ 500       ┆ … ┆ 0      ┆ 0.904 ±   ┆ 3.261 ±   ┆ 4.911 ±   │
│            ┆ r          ┆           ┆           ┆   ┆        ┆ 0.002     ┆ 0.027     ┆ 0.044     │
└────────────┴────────────┴───────────┴───────────┴───┴────────┴───────────┴───────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x10af17680>, _boxhead=Boxhead([ColInfo(var='panel', type=<ColInfoTypeEnum.row_group: 3>, column_label='panel', column_align='left', column_width=None), ColInfo(var='method', type=<ColInfoTypeEnum.stub: 2>, column_label='method', column_align='left', column_width=None), ColInfo(var='selection_n', type=<ColInfoTypeEnum.default: 1>, column_label='selection_n', column_align='right', column_width=None), ColInfo(var='band_fit_n', type=<ColInfoTypeEnum.default: 1>, column_label='band_fit_n', column_align='right', column_width=None), ColInfo(var='complete', type=<ColInfoTypeEnum.default: 1>, column_label='complete', column_align='right', column_width=None), ColInfo(var='failed', type=<ColInfoTypeEnum.default: 1>, column_label='failed', column_align='right', column_width=None), ColInfo(var='coverage', type=<ColInfoTypeEnum.default: 1>, column_label='coverage', column_align='left', column_width=None), ColInfo(var='width', type=<ColInfoTypeEnum.default: 1>, column_label='width', column_align='left', column_width=None), ColInfo(var='interval_score', type=<ColInfoTypeEnum.default: 1>, column_la

## Calibration boundary

A split-conformal prediction rule must be fixed without calibration responses. Refitting a formula chosen on those responses does not undo their use. The original `discover` selected its family, transforms, terms and stopping on all development rows, then passed that model to an internal calibration split. The new path reserves one half before any response-dependent discovery stage and calibrates its selected model without another refit.

Calls with `newdata` request prediction bands and therefore reserve this holdout by default. Calls without `newdata` keep all supplied analysis rows and attach no prediction-band guarantee. A caller can supply independently reserved `calibration=` rows to use every `data` row for selection. The returned model and any REML effect summaries use the selection rows. These effect summaries remain conditional on the selected model; the prediction-band construction does not solve post-selection effect inference [#464](https://github.com/fcela/mboostrs/issues/464).

Exchangeable observations have a joint distribution unchanged by row order; independent draws from one population satisfy this condition. The marginal coverage statement assumes exchangeable observations and a prediction rule independent of calibration responses. It is not a guarantee for each covariate value or for clustered, time-ordered or shifted data. Caller-supplied calibration rows remain the caller's independence claim; their input namespace and positions are recorded separately.

In [4]:
import mboostrs as mb
rng = np.random.default_rng(631)
x1, x2 = rng.uniform(-1, 1, (2, 600))
y = np.sin(2*np.pi*x1)+.5*x2+(.3+1.5*np.abs(x1))*rng.normal(size=600)
data = pl.DataFrame({"x1": x1, "x2": x2, "y": y})
query = pl.DataFrame({"x1": [-.8, 0., .8], "x2": [0., 0., 0.]})

# Family, transforms, formula and stopping read only the reserved selection half.
result = mb.discover(data, "y", newdata=query, seed=17, level=.9)
parts = result.provenance["partition"]
assert set(parts["selection_rows"]).isdisjoint(parts["calibration_rows"])
selection_count = len(parts["selection_rows"])
calibration_count = len(parts["calibration_rows"])
assert result.interval is not None
report = pl.DataFrame({"quantity": ["family", "mean formula", "scale formula", "stopping", "selection rows", "calibration rows"],
    "value": [result.family, result.formula, str(result.spread_formula), str(result.mstop),
              str(selection_count), str(calibration_count)]})
display(GT(report).tab_header(title="Selection-only analysis",
    subtitle="Call seed 17; calibration responses cannot change these fitted choices"))


GT(_tbl_data=shape: (6, 2)
┌──────────────────┬─────────────────────────────────┐
│ quantity         ┆ value                           │
│ ---              ┆ ---                             │
│ str              ┆ str                             │
╞══════════════════╪═════════════════════════════════╡
│ family           ┆ Gaussian                        │
│ mean formula     ┆ bbs(x1, df=4.0) + bbs(x2, df=4… │
│ scale formula    ┆ bbs(x1, df=4)                   │
│ stopping         ┆ 381                             │
│ selection rows   ┆ 300                             │
│ calibration rows ┆ 300                             │
└──────────────────┴─────────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x1242edbb0>, _boxhead=Boxhead([ColInfo(var='quantity', type=<ColInfoTypeEnum.default: 1>, column_label='quantity', column_align='left', column_width=None), ColInfo(var='value', type=<ColInfoTypeEnum.default: 1>, column_label='value', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x126245d30>, _spanners=Spanners([]), _heading=Heading(title='Selection-only analysis', subtitle='Call seed 17; calibration responses cannot change these fitted choices', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x120e08440>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x120e0bbc0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x125420140>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', va

In [5]:
display(GT(query.hstack(result.predictions.drop("row"))).tab_header(
    title="Predictions from the selected fit and its calibrated band",
    subtitle="Response units; the held-out responses determine band endpoints")
    .fmt_number(columns=[name for name in result.predictions.columns if name != "row"]+["x1", "x2"], decimals=3)
    .tab_source_note("Sigma is included only when discovery selects a location-scale model. Mean and sigma are from the selection-only fit. The interval result separately records its assumptions, calibration source and marginal bound."))


GT(_tbl_data=shape: (3, 6)
┌──────┬─────┬───────────┬──────────┬───────────┬──────────┐
│ x1   ┆ x2  ┆ mean      ┆ sigma    ┆ lower     ┆ upper    │
│ ---  ┆ --- ┆ ---       ┆ ---      ┆ ---       ┆ ---      │
│ f64  ┆ f64 ┆ f64       ┆ f64      ┆ f64       ┆ f64      │
╞══════╪═════╪═══════════╪══════════╪═══════════╪══════════╡
│ -0.8 ┆ 0.0 ┆ 0.724072  ┆ 1.522383 ┆ -1.776592 ┆ 3.224736 │
│ 0.0  ┆ 0.0 ┆ 0.013604  ┆ 0.415337 ┆ -0.666132 ┆ 0.69334  │
│ 0.8  ┆ 0.0 ┆ -1.073968 ┆ 1.380699 ┆ -3.341582 ┆ 1.193647 │
└──────┴─────┴───────────┴──────────┴───────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x12538cf20>, _boxhead=Boxhead([ColInfo(var='x1', type=<ColInfoTypeEnum.default: 1>, column_label='x1', column_align='right', column_width=None), ColInfo(var='x2', type=<ColInfoTypeEnum.default: 1>, column_label='x2', column_align='right', column_width=None), ColInfo(var='mean', type=<ColInfoTypeEnum.default: 1>, column_label='mean', column_align='right', column_width=None), ColInfo(var='sigma', type=<ColInfoTypeEnum.default: 1>, column_label='sigma', column_align='right', column_width=None), ColInfo(var='lower', type=<ColInfoTypeEnum.default: 1>, column_label='lower', column_align='right', column_width=None), ColInfo(var='upper', type=<ColInfoTypeEnum.default: 1>, column_label='upper', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x109b61fa0>, _spanners=Spanners([]), _heading=Heading(title='Predictions from the selected fit and its calibrated band', subtitle='Response units; the held-out responses determine band endpoints', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12431e9c0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1253c2bd0>, _source_notes=['Sigma is included only when discovery selects a location-scale model. Mean and sigma are from the selection-only fit. The interval result separately records its assumptions, calibration source and marginal bound.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1253c09e0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x125bdbce0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_b

The call uses {eval}`selection_count` rows to select and fit the model, with {eval}`calibration_count` rows held out from that analysis. `result.interval` exposes the calibration count, coverage assumptions and source. The same chosen model supplies the reported mean and any fitted sigma; it is not replaced by an all-data fit after calibration.

The automatic split uses the existing Rust two-fold random assignment. Its exact row positions are in `result.provenance["partition"]`, before predictor hygiene and missing-data handling. The following call supplies that same partition explicitly. In an application the explicit frame must have been reserved before any model choice.

In [6]:
selection = data[parts["selection_rows"]]
calibration = data[parts["calibration_rows"]]
explicit = mb.discover(selection, "y", calibration=calibration, newdata=query, seed=17, level=.9)
np.testing.assert_array_equal(result.predictions.to_numpy(), explicit.predictions.to_numpy())
assert explicit.provenance["partition"]["calibration_input"] == "calibration"

# A Gaussian location-scale fit models the mean and standard deviation.
# This supplied reference uses 100 updates per parameter, with nu=0.1.
known = mb.gamboostLSS({"mu": "y ~ bbs(x1, df=4) + bols(x2)", "sigma": "y ~ bbs(x1, df=4)"},
    selection, family=mb.GaussianLSS(), control=mb.boost_control(100))
known_band = mb.conformal_interval(known, query, calibration=calibration, level=.9, extrapolation="clip")
assert known_band.n_training == selection_count
assert known_band.n_calibration == calibration_count


## Frozen comparison

Each panel has 50 independent development/test draws, with 1,000 untouched test rows per draw. Development size is 1,000 except the sharp-interaction panel (800) and small-sample panel (80). The small panel therefore leaves 40 selection and 40 calibration rows. Selection still requires 30 complete rows; an inadequate automatic split raises a clear error rather than weakening that minimum.

The absolute, constant, null and heavy-tail panels use independent uniform predictors on `[-1,1]`. The smooth mean is `sin(2*pi*x1)+0.5*x2`; the null mean is zero. Absolute scale is `0.3+1.5*abs(x1)`, otherwise one. Heavy-tail noise is unit-variance Student t with three degrees of freedom. The sharp panel has mean `3*sign(x1)*sign(x2)+x3`, noise SD 0.5 and two unused predictors. The correlated panel uses standard-normal predictors with correlation 0.9, linear mean `0.8*x1+0.4*x2` and noise SD 0.5. The Poisson mean is `exp(0.3+0.6*x1-0.4*x2)`.

Known-specification fits receive the stated mean structure and family, plus the scale covariate only on the absolute panel. Their stopping is fixed at 100 updates. Discovery receives no structure. CatBoost split/CQR use 500 depth-six trees at rate 0.05; LightGBM split/CQR use 300 trees with 15 leaves at rate 0.05; sklearn CQR uses 300 depth-three trees at rate 0.05. Controls and the new known/discovery arms share the exact Rust partition. All settings are untuned and use one thread.

The old discovery arm retains its original later NumPy split. It differs in selection data use and partition, so the paired comparison measures the complete procedure, not a causal effect of the reservation operation alone. CQR point forecasts are band midpoints; point MSE evaluates those reported forecasts against observed responses without claiming that every midpoint estimates a conditional mean.

### Competitor construction

The following extension shows the CatBoost CQR control on the small public-API example above; it is not an extra confirmation draw. The two fixed quantile fits use selection rows only. Their calibration scores measure how far a held-out response falls outside the fitted 5%–95% band. The finite-sample order statistic uses the ceiling of `(n_calibration + 1) * level`; a negative correction shrinks a conservative band and can produce an empty set. The benchmark retains those empty sets.

In [7]:
from catboost import CatBoostRegressor
selection_x = selection.drop("y").to_numpy()
calibration_x = calibration.drop("y").to_numpy()
query_x = query.to_numpy()
control_settings = dict(iterations=500, depth=6, learning_rate=.05, random_seed=17,
                        verbose=False, thread_count=1, allow_writing_files=False)

# Both quantile models are fixed before any calibration response is read.
low_model = CatBoostRegressor(loss_function="Quantile:alpha=0.05", **control_settings)
high_model = CatBoostRegressor(loss_function="Quantile:alpha=0.95", **control_settings)
low_model.fit(selection_x, selection["y"].to_numpy())
high_model.fit(selection_x, selection["y"].to_numpy())


CatBoostRegressor(allow_writing_files=False, depth=6, iterations=500, learning_rate=0.05, loss_function='Quantile:alpha=0.95', random_seed=17, thread_count=1, verbose=False)

In [8]:
held_y = calibration["y"].to_numpy()
scores = np.maximum(low_model.predict(calibration_x)-held_y,
                    held_y-high_model.predict(calibration_x))
rank = int(np.ceil((len(scores)+1)*.9))
correction = np.sort(scores)[rank-1]
control_low = low_model.predict(query_x)-correction
control_high = high_model.predict(query_x)+correction
control = query.with_columns(pl.Series("lower", control_low), pl.Series("upper", control_high),
                             pl.Series("empty", control_high < control_low))
display(GT(control).tab_header(title="Example CatBoost CQR construction",
    subtitle="Response-unit bounds; the held-out order statistic supplies the correction")
    .fmt_number(columns=["x1", "x2", "lower", "upper"], decimals=3)
    .tab_source_note("This example uses 300 selection and 300 calibration rows. The confirmation controls use their panel's frozen partition; no test outcome chooses the settings or correction."))


GT(_tbl_data=shape: (3, 5)
┌──────┬─────┬───────────┬──────────┬───────┐
│ x1   ┆ x2  ┆ lower     ┆ upper    ┆ empty │
│ ---  ┆ --- ┆ ---       ┆ ---      ┆ ---   │
│ f64  ┆ f64 ┆ f64       ┆ f64      ┆ bool  │
╞══════╪═════╪═══════════╪══════════╪═══════╡
│ -0.8 ┆ 0.0 ┆ -2.06462  ┆ 2.839586 ┆ false │
│ 0.0  ┆ 0.0 ┆ -2.073929 ┆ 1.786313 ┆ false │
│ 0.8  ┆ 0.0 ┆ -4.41532  ┆ 2.216456 ┆ false │
└──────┴─────┴───────────┴──────────┴───────┘, _body=<great_tables._gt_data.Body object at 0x10970b3b0>, _boxhead=Boxhead([ColInfo(var='x1', type=<ColInfoTypeEnum.default: 1>, column_label='x1', column_align='right', column_width=None), ColInfo(var='x2', type=<ColInfoTypeEnum.default: 1>, column_label='x2', column_align='right', column_width=None), ColInfo(var='lower', type=<ColInfoTypeEnum.default: 1>, column_label='lower', column_align='right', column_width=None), ColInfo(var='upper', type=<ColInfoTypeEnum.default: 1>, column_label='upper', column_align='right', column_width=None), ColInfo(var='empty', type=<ColInfoTypeEnum.default: 1>, column_label='empty', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x12d5b8d40>, _spanners=Spanners([]), _heading=Heading(title='Example CatBoost CQR construction', subtitle='Response-unit bounds; the held-out order statistic supplies the correction', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12cb74290>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12d5efaa0>, _source_notes=["This example uses 300 selection and 300 calibration rows. The confirmation controls use their panel's frozen partition; no test outcome chooses the settings or correction."], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x12d6646b0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x12d4c4cb0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=O

In [9]:
lookup = {(r["panel"], r["seed"], r["method"], r["phase"]): r for r in records}
paired_rows = []
for panel in panels:
    pairs = [(lookup[panel, seed, "discovery", "baseline"], lookup[panel, seed, "discovery", "fixed"])
             for seed in range(6311000, 6311050)]
    complete = [(a, b) for a, b in pairs if a["status"] == b["status"] == "ok"]
    assert all(a["input_digest"] == b["input_digest"] for a, b in complete)
    row = {"panel": panel, "complete_pairs": len(complete)}
    values = {"coverage_points": [100*(b["coverage"]-a["coverage"]) for a,b in complete],
        "width_percent": [100*(b["width"]/a["width"]-1) for a,b in complete],
        "score_change": [b["interval_score"]-a["interval_score"] for a,b in complete],
        "point_mse_change": [b["point_mse"]-a["point_mse"] for a,b in complete]}
    for metric, vector in values.items():
        row[metric], row[metric+"_se"] = mean_se(vector) if vector else (None, None)
    paired_rows.append(row)
paired_frame = pl.DataFrame(paired_rows)
display(GT(paired_frame).tab_header(title="Paired change from adaptive to reserved calibration",
    subtitle="New minus baseline; lower width/score/error is better; coverage change in percentage points")
    .fmt_number(columns=[name for name in paired_frame.columns if name not in ("panel", "complete_pairs")], decimals=3)
    .tab_source_note("Monte Carlo SE columns use independent data draws, paired within each draw. Missing pairs are counted, never replaced. Reservation can change both the selected structure and its estimation sample."))


GT(_tbl_data=shape: (8, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ panel     ┆ complete_ ┆ coverage_ ┆ coverage_ ┆ … ┆ score_cha ┆ score_cha ┆ point_mse ┆ point_ms │
│ ---       ┆ pairs     ┆ points    ┆ points_se ┆   ┆ nge       ┆ nge_se    ┆ _change   ┆ e_change │
│ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ _se      │
│           ┆ i64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ absolute  ┆ 50        ┆ -0.1      ┆ 0.26388   ┆ … ┆ -0.162065 ┆ 0.017914  ┆ 0.043677  ┆ 0.006556 │
│ constant  ┆ 50        ┆ 0.24      ┆ 0.205178  ┆ … ┆ 0.019614  ┆ 0.010215  ┆ 0.021156  ┆ 0.004387 │
│ null      ┆ 50        ┆ 0.056     ┆ 0.200554  ┆ … ┆ -0.003263 ┆ 0.003183  ┆ 0.001216  ┆ 0.000509 │
│ sharp     ┆ 50        ┆ -0.05     ┆ 0.270966  ┆ … ┆ -0.176177 ┆ 0.165367  ┆ 0.100673  ┆ 0.087058 │
│ small     ┆ 50        ┆ 0.326     ┆ 0.755296  ┆ … ┆ 0.090275  ┆ 0.082277  ┆ 0.222074  ┆ 0.038231 │
│ correlate ┆ 50        ┆ 0.056     ┆ 0.222585  ┆ … ┆ 0.003822  ┆ 0.0046    ┆ 0.002348  ┆ 0.000922 │
│ d         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ poisson   ┆ 50        ┆ -0.268    ┆ 0.2094    ┆ … ┆ 0.0425    ┆ 0.016595  ┆ 0.038182  ┆ 0.008036 │
│ heavy_tai ┆ 50        ┆ 0.06      ┆ 0.188766  ┆ … ┆ 0.003546  ┆ 0.011472  ┆ 0.018375  ┆ 0.004557 │
│ l         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
└───────────┴───────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x120d59bb0>, _boxhead=Boxhead([ColInfo(var='panel', type=<ColInfoTypeEnum.default: 1>, column_label='panel', column_align='left', column_width=None), ColInfo(var='complete_pairs', type=<ColInfoTypeEnum.default: 1>, column_label='complete_pairs', column_align='right', column_width=None), ColInfo(var='coverage_points', type=<ColInfoTypeEnum.default: 1>, column_label='coverage_points', column_align='right', column_width=None), ColInfo(var='coverage_points_se', type=<ColInfoTypeEnum.default: 1>, column_label='coverage_points_se', column_align='right', column_width=None), ColInfo(var='width_percent', type=<ColInfoTypeEnum.default: 1>, column_label='width_percent', column_align='right', column_width=None), ColInfo(var='width_percent_se', type=<ColInfoTypeEnum.default: 1>, column_label='width_percent_se', column_align='right', column_width=None), ColInfo(var='score_change', type=<ColInfoTypeEnum.default: 1>, column_label='score_change', column_align='right', column_width=None), ColInfo(var='score_change_se', type=<ColInfoTypeEnum.default: 1>, column_label='score_change_se', column_align='right', column_width=None), ColInfo(var='point_mse_change', type=<ColInfoTypeEnum.default: 1>, column_label='point_mse_change', column_align='right', column_width=None), ColInfo(var='point_mse_change_se', type=<ColInfoTypeEnum.default: 1>, column_label='point_mse_change_se', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x12d0ae060>, _spanners=Spanners([]), _heading=Heading(title='Paired change from adaptive to reserved calibration', subtitle='New minus baseline; lower width/score/error is better; coverage change in percentage points', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12d51b080>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12613daf0>, _source_notes=['Monte Carlo SE columns use independent data draws, paired within each draw. Missing pairs are counted, never replaced. Reservation can change both the selected structure and its estimation sample.'], _footnotes=[], _styles=[]

In [10]:
gap_rows = []
for panel in panels:
    best = leaders[panel]
    for phase, method in [("baseline", "discovery"), ("fixed", "discovery"), ("fixed", "known")]:
        differences = []
        if best is not None:
            for seed in range(6311000, 6311050):
                row, peer = lookup[panel, seed, method, phase], lookup[panel, seed, best, "fixed"]
                if row["status"] == peer["status"] == "ok":
                    assert row["input_digest"] == peer["input_digest"]
                    differences.append(100*(row["width"]/peer["width"]-1))
        mean, se = mean_se(differences) if differences else (None, None)
        gap_rows.append({"panel": panel, "track": method+"/"+phase, "retrospective_peer": best,
                         "paired_draws": len(differences), "width_gap_pct": mean, "MC_SE_points": se})
gap_frame = pl.DataFrame(gap_rows)
display(GT(gap_frame, groupname_col="panel").tab_header(title="Gaps to retrospective eligible competitors",
    subtitle="Percent of competitor width; negative favors mboostrs; known and discovery shown separately")
    .fmt_number(columns=["width_gap_pct", "MC_SE_points"], decimals=2)
    .tab_source_note("Winner selected from complete competitor test results with mean coverage in [0.88,0.92]; small-n upper tolerance adds 1/41 for quantization. No eligible competitor means no width gap. SE does not account for retrospective winner selection. A sub-1% point estimate is not a population bound."))


GT(_tbl_data=shape: (24, 6)
┌────────────┬───────────────────┬───────────────────┬──────────────┬───────────────┬──────────────┐
│ panel      ┆ track             ┆ retrospective_pee ┆ paired_draws ┆ width_gap_pct ┆ MC_SE_points │
│ ---        ┆ ---               ┆ r                 ┆ ---          ┆ ---           ┆ ---          │
│ str        ┆ str               ┆ ---               ┆ i64          ┆ f64           ┆ f64          │
│            ┆                   ┆ str               ┆              ┆               ┆              │
╞════════════╪═══════════════════╪═══════════════════╪══════════════╪═══════════════╪══════════════╡
│ absolute   ┆ discovery/baselin ┆ lightgbm_cqr      ┆ 50           ┆ 0.795285      ┆ 0.839013     │
│            ┆ e                 ┆                   ┆              ┆               ┆              │
│ absolute   ┆ discovery/fixed   ┆ lightgbm_cqr      ┆ 50           ┆ -2.919276     ┆ 0.546675     │
│ absolute   ┆ known/fixed       ┆ lightgbm_cqr      ┆ 50           ┆ 1.023073      ┆ 0.547556     │
│ constant   ┆ discovery/baselin ┆ catboost_split    ┆ 50           ┆ -5.800741     ┆ 0.668716     │
│            ┆ e                 ┆                   ┆              ┆               ┆              │
│ constant   ┆ discovery/fixed   ┆ catboost_split    ┆ 50           ┆ -4.781988     ┆ 0.458624     │
│ …          ┆ …                 ┆ …                 ┆ …            ┆ …             ┆ …            │
│ poisson    ┆ discovery/fixed   ┆ lightgbm_cqr      ┆ 50           ┆ 0.117286      ┆ 1.495041     │
│ poisson    ┆ known/fixed       ┆ lightgbm_cqr      ┆ 50           ┆ 4.429089      ┆ 1.515845     │
│ heavy_tail ┆ discovery/baselin ┆ catboost_split    ┆ 50           ┆ -6.136947     ┆ 0.876971     │
│            ┆ e                 ┆                   ┆              ┆               ┆              │
│ heavy_tail ┆ discovery/fixed   ┆ catboost_split    ┆ 50           ┆ -6.093813     ┆ 0.552318     │
│ heavy_tail ┆ known/fixed       ┆ catboost_split    ┆ 50           ┆ 0.331235      ┆ 0.698882     │
└────────────┴───────────────────┴───────────────────┴──────────────┴───────────────┴──────────────┘, _body=<great_tables._gt_data.Body object at 0x12da2b410>, _boxhead=Boxhead([ColInfo(var='panel', type=<ColInfoTypeEnum.row_group: 3>, column_label='panel', column_align='left', column_width=None), ColInfo(var='track', type=<ColInfoTypeEnum.default: 1>, column_label='track', column_align='left', column_width=None), ColInfo(var='retrospective_peer', type=<ColInfoTypeEnum.default: 1>, column_label='retrospective_peer', column_align='left', column_width=None), ColInfo(var='paired_draws', type=<ColInfoTypeEnum.default: 1>, column_label='paired_draws', column_align='right', column_width=None), ColInfo(var='width_gap_pct', type=<ColInfoTypeEnum.default: 1>, column_label='width_gap_pct', column_align='right', column_width=None), ColInfo(var='MC_SE_points', type=<ColInfoTypeEnum.default: 1>, column_label='MC_SE_points', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x126245d30>, _spanners=Spanners([]), _heading=Heading(title='Gaps to retrospective eligible competitors', subtitle='Percent of competitor width; negative favors mboostrs; known and discovery shown separately', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12da2bdd0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12da2be60>, _source_notes=['Winner selected from complete competitor test results with mean coverage in [0.88,0.92]; small-n upper tolerance adds 1/41 for quantization. No eligible competitor means no width gap. SE does not account for retrospective winner selection. A sub-1% point estimate is not a population bound.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x12da2afc0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x12c0a0380>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id

In [11]:
conditional = summary_frame.select("panel", "method", "coverage", "coverage_se", "coverage_low", "coverage_high", "empty_sets", "failed")
display(GT(conditional, groupname_col="panel").tab_header(title="Marginal and conditional coverage",
    subtitle="Shares of test responses covered; nominal 0.90; 50,000 test predictions per complete row")
    .fmt_number(columns=["coverage", "coverage_se", "coverage_low", "coverage_high"], decimals=3)
    .tab_source_note("Low/high groups are thirds of true test noise scale, available only to the evaluator. Constant-scale panels put all rows in both groups. Empty CQR sets have zero width and no coverage; their two-quantile score is retained."))


GT(_tbl_data=shape: (64, 8)
┌────────────┬─────────────┬──────────┬────────────┬────────────┬────────────┬────────────┬────────┐
│ panel      ┆ method      ┆ coverage ┆ coverage_s ┆ coverage_l ┆ coverage_h ┆ empty_sets ┆ failed │
│ ---        ┆ ---         ┆ ---      ┆ e          ┆ ow         ┆ igh        ┆ ---        ┆ ---    │
│ str        ┆ str         ┆ f64      ┆ ---        ┆ ---        ┆ ---        ┆ i64        ┆ i64    │
│            ┆             ┆          ┆ f64        ┆ f64        ┆ f64        ┆            ┆        │
╞════════════╪═════════════╪══════════╪════════════╪════════════╪════════════╪════════════╪════════╡
│ absolute   ┆ discovery/b ┆ 0.90024  ┆ 0.002489   ┆ 0.903114   ┆ 0.902515   ┆ 0          ┆ 0      │
│            ┆ aseline     ┆          ┆            ┆            ┆            ┆            ┆        │
│ absolute   ┆ discovery/f ┆ 0.89924  ┆ 0.002205   ┆ 0.898802   ┆ 0.894072   ┆ 0          ┆ 0      │
│            ┆ ixed        ┆          ┆            ┆            ┆            ┆            ┆        │
│ absolute   ┆ known       ┆ 0.9017   ┆ 0.002358   ┆ 0.906946   ┆ 0.902934   ┆ 0          ┆ 0      │
│ absolute   ┆ catboost_sp ┆ 0.90374  ┆ 0.00247    ┆ 0.997006   ┆ 0.779222   ┆ 0          ┆ 0      │
│            ┆ lit         ┆          ┆            ┆            ┆            ┆            ┆        │
│ absolute   ┆ catboost_cq ┆ 0.90272  ┆ 0.002702   ┆ 0.966647   ┆ 0.82024    ┆ 84         ┆ 0      │
│            ┆ r           ┆          ┆            ┆            ┆            ┆            ┆        │
│ …          ┆ …           ┆ …        ┆ …          ┆ …          ┆ …          ┆ …          ┆ …      │
│ heavy_tail ┆ catboost_sp ┆ 0.9019   ┆ 0.002278   ┆ 0.9019     ┆ 0.9019     ┆ 0          ┆ 0      │
│            ┆ lit         ┆          ┆            ┆            ┆            ┆            ┆        │
│ heavy_tail ┆ catboost_cq ┆ 0.9029   ┆ 0.002483   ┆ 0.9029     ┆ 0.9029     ┆ 57         ┆ 0      │
│            ┆ r           ┆          ┆            ┆            ┆            ┆            ┆        │
│ heavy_tail ┆ lightgbm_sp ┆ 0.90216  ┆ 0.00209    ┆ 0.90216    ┆ 0.90216    ┆ 0          ┆ 0      │
│            ┆ lit         ┆          ┆            ┆            ┆            ┆            ┆        │
│ heavy_tail ┆ lightgbm_cq ┆ 0.90436  ┆ 0.002319   ┆ 0.90436    ┆ 0.90436    ┆ 0          ┆ 0      │
│            ┆ r           ┆          ┆            ┆            ┆            ┆            ┆        │
│ heavy_tail ┆ sklearn_cqr ┆ 0.90406  ┆ 0.001966   ┆ 0.90406    ┆ 0.90406    ┆ 3          ┆ 0      │
└────────────┴─────────────┴──────────┴────────────┴────────────┴────────────┴────────────┴────────┘, _body=<great_tables._gt_data.Body object at 0x12da2b920>, _boxhead=Boxhead([ColInfo(var='panel', type=<ColInfoTypeEnum.row_group: 3>, column_label='panel', column_align='left', column_width=None), ColInfo(var='method', type=<ColInfoTypeEnum.default: 1>, column_label='method', column_align='left', column_width=None), ColInfo(var='coverage', type=<ColInfoTypeEnum.default: 1>, column_label='coverage', column_align='right', column_width=None), ColInfo(var='coverage_se', type=<ColInfoTypeEnum.default: 1>, column_label='coverage_se', column_align='right', column_width=None), ColInfo(var='coverage_low', type=<ColInfoTypeEnum.default: 1>, column_label='coverage_low', column_align='right', column_width=None), ColInfo(var='coverage_high', type=<ColInfoTypeEnum.default: 1>, column_label='coverage_high', column_align='right', column_width=None), ColInfo(var='empty_sets', type=<ColInfoTypeEnum.default: 1>, column_label='empty_sets', column_align='right', column_width=None), ColInfo(var='failed', type=<ColInfoTypeEnum.default: 1>, column_label='failed', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x120d5a480>, _spanners=Spanners([]), _heading=Heading(title='Marginal and conditional coverage', subtitle='Shares of test responses covered; nominal 0.90; 50,000 test predictions per complete row', pr

In [12]:
selection_rows = []
for panel in panels:
    for phase, source in [("baseline", baseline), ("fixed", fixed)]:
        rows = [r for r in source["results"] if r["panel"] == panel and r["method"] == "discovery" and r["status"] == "ok"]
        selection_rows.append({"panel": panel, "phase": phase, "completed": len(rows),
            "mean_terms_selected": sum(r["details"]["formula"] != "1" for r in rows),
            "scale_selected": sum(r["details"]["spread_formula"] is not None for r in rows),
            "families": json.dumps(sorted(set(r["details"]["family"] for r in rows)))})
display(GT(pl.DataFrame(selection_rows), groupname_col="panel").tab_header(
    title="Discovery outcomes after reserving calibration",
    subtitle="Counts out of 50 independent draws; Gaussian scale selection is distinct from mean selection")
    .tab_source_note("Null, constant-scale and sharp-mean controls expose inappropriate terms or mean-versus-scale ambiguity. A held-out prediction guarantee does not calibrate effect-selection error. All outcomes remain visible."))


GT(_tbl_data=shape: (16, 6)
┌────────────┬──────────┬───────────┬─────────────────────┬────────────────┬──────────────┐
│ panel      ┆ phase    ┆ completed ┆ mean_terms_selected ┆ scale_selected ┆ families     │
│ ---        ┆ ---      ┆ ---       ┆ ---                 ┆ ---            ┆ ---          │
│ str        ┆ str      ┆ i64       ┆ i64                 ┆ i64            ┆ str          │
╞════════════╪══════════╪═══════════╪═════════════════════╪════════════════╪══════════════╡
│ absolute   ┆ baseline ┆ 50        ┆ 50                  ┆ 50             ┆ ["Gaussian"] │
│ absolute   ┆ fixed    ┆ 50        ┆ 50                  ┆ 50             ┆ ["Gaussian"] │
│ constant   ┆ baseline ┆ 50        ┆ 50                  ┆ 0              ┆ ["Gaussian"] │
│ constant   ┆ fixed    ┆ 50        ┆ 50                  ┆ 0              ┆ ["Gaussian"] │
│ null       ┆ baseline ┆ 50        ┆ 0                   ┆ 0              ┆ ["Gaussian"] │
│ …          ┆ …        ┆ …         ┆ …                   ┆ …              ┆ …            │
│ correlated ┆ fixed    ┆ 50        ┆ 50                  ┆ 0              ┆ ["Gaussian"] │
│ poisson    ┆ baseline ┆ 50        ┆ 50                  ┆ 0              ┆ ["Poisson"]  │
│ poisson    ┆ fixed    ┆ 50        ┆ 49                  ┆ 0              ┆ ["Poisson"]  │
│ heavy_tail ┆ baseline ┆ 50        ┆ 50                  ┆ 2              ┆ ["Gaussian"] │
│ heavy_tail ┆ fixed    ┆ 50        ┆ 50                  ┆ 0              ┆ ["Gaussian"] │
└────────────┴──────────┴───────────┴─────────────────────┴────────────────┴──────────────┘, _body=<great_tables._gt_data.Body object at 0x11579cb30>, _boxhead=Boxhead([ColInfo(var='panel', type=<ColInfoTypeEnum.row_group: 3>, column_label='panel', column_align='left', column_width=None), ColInfo(var='phase', type=<ColInfoTypeEnum.default: 1>, column_label='phase', column_align='left', column_width=None), ColInfo(var='completed', type=<ColInfoTypeEnum.default: 1>, column_label='completed', column_align='right', column_width=None), ColInfo(var='mean_terms_selected', type=<ColInfoTypeEnum.default: 1>, column_label='mean_terms_selected', column_align='right', column_width=None), ColInfo(var='scale_selected', type=<ColInfoTypeEnum.default: 1>, column_label='scale_selected', column_align='right', column_width=None), ColInfo(var='families', type=<ColInfoTypeEnum.default: 1>, column_label='families', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x12da2bb60>, _spanners=Spanners([]), _heading=Heading(title='Discovery outcomes after reserving calibration', subtitle='Counts out of 50 independent draws; Gaussian scale selection is distinct from mean selection', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x111863410>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x111863ef0>, _source_notes=['Null, constant-scale and sharp-mean controls expose inappropriate terms or mean-versus-scale ambiguity. A held-out prediction guarantee does not calibrate effect-selection error. All outcomes remain visible.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x12da2b3e0>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(

In [13]:
cost = summary_frame.select("panel", "method", "elapsed", "elapsed_se", "point_mse", "point_mse_se")
display(GT(cost, groupname_col="panel").tab_header(title="Recorded cost and point prediction error",
    subtitle="Elapsed seconds and squared response-unit error; means and Monte Carlo SE")
    .fmt_number(columns=["elapsed", "elapsed_se", "point_mse", "point_mse_se"], decimals=3)
    .tab_source_note("Whole workflow: selection/fitting, calibration and prediction. Timings share a host, include result wrapping and exclude checkpoint-file writing; they are descriptive costs, not controlled implementation speed ratios. CQR point forecasts are interval midpoints."))


GT(_tbl_data=shape: (64, 6)
┌────────────┬────────────────────┬──────────┬────────────┬───────────┬──────────────┐
│ panel      ┆ method             ┆ elapsed  ┆ elapsed_se ┆ point_mse ┆ point_mse_se │
│ ---        ┆ ---                ┆ ---      ┆ ---        ┆ ---       ┆ ---          │
│ str        ┆ str                ┆ f64      ┆ f64        ┆ f64       ┆ f64          │
╞════════════╪════════════════════╪══════════╪════════════╪═══════════╪══════════════╡
│ absolute   ┆ discovery/baseline ┆ 0.762237 ┆ 0.019258   ┆ 1.316179  ┆ 0.009482     │
│ absolute   ┆ discovery/fixed    ┆ 0.320621 ┆ 0.014353   ┆ 1.359856  ┆ 0.012433     │
│ absolute   ┆ known              ┆ 0.003655 ┆ 0.000024   ┆ 1.4642    ┆ 0.011378     │
│ absolute   ┆ catboost_split     ┆ 0.137275 ┆ 0.014071   ┆ 1.548813  ┆ 0.017137     │
│ absolute   ┆ catboost_cqr       ┆ 0.251208 ┆ 0.000338   ┆ 1.495747  ┆ 0.013891     │
│ …          ┆ …                  ┆ …        ┆ …          ┆ …         ┆ …            │
│ heavy_tail ┆ catboost_split     ┆ 0.124554 ┆ 0.000108   ┆ 1.116427  ┆ 0.027598     │
│ heavy_tail ┆ catboost_cqr       ┆ 0.252232 ┆ 0.000363   ┆ 1.168801  ┆ 0.027856     │
│ heavy_tail ┆ lightgbm_split     ┆ 0.031752 ┆ 0.000198   ┆ 1.162831  ┆ 0.027165     │
│ heavy_tail ┆ lightgbm_cqr       ┆ 0.057158 ┆ 0.000153   ┆ 1.144176  ┆ 0.028219     │
│ heavy_tail ┆ sklearn_cqr        ┆ 0.260545 ┆ 0.001697   ┆ 1.219125  ┆ 0.030097     │
└────────────┴────────────────────┴──────────┴────────────┴───────────┴──────────────┘, _body=<great_tables._gt_data.Body object at 0x12da2bc50>, _boxhead=Boxhead([ColInfo(var='panel', type=<ColInfoTypeEnum.row_group: 3>, column_label='panel', column_align='left', column_width=None), ColInfo(var='method', type=<ColInfoTypeEnum.default: 1>, column_label='method', column_align='left', column_width=None), ColInfo(var='elapsed', type=<ColInfoTypeEnum.default: 1>, column_label='elapsed', column_align='right', column_width=None), ColInfo(var='elapsed_se', type=<ColInfoTypeEnum.default: 1>, column_label='elapsed_se', column_align='right', column_width=None), ColInfo(var='point_mse', type=<ColInfoTypeEnum.default: 1>, column_label='point_mse', column_align='right', column_width=None), ColInfo(var='point_mse_se', type=<ColInfoTypeEnum.default: 1>, column_label='point_mse_se', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1118616d0>, _spanners=Spanners([]), _heading=Heading(title='Recorded cost and point prediction error', subtitle='Elapsed seconds and squared response-unit error; means and Monte Carlo SE', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12da70da0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12da70dd0>, _source_notes=['Whole workflow: selection/fitting, calibration and prediction. Timings share a host, include result wrapping and exclude checkpoint-file writing; they are descriptive costs, not controlled implementation speed ratios. CQR point forecasts are interval midpoints.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x12da70d70>, _formats=[<great_tables._gt_data.FormatInfo object at 0x111863410>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category=

In [14]:
paired_by_panel = {row["panel"]: row for row in paired_rows}
new_summaries = [row for row in summary if row["method"] == "discovery/fixed"]
minimum_new_coverage = min(row["coverage"] for row in new_summaries)
maximum_new_coverage = max(row["coverage"] for row in new_summaries)
absolute_change = paired_by_panel["absolute"]
small_change = paired_by_panel["small"]
known_poisson = next(row for row in gap_rows if row["panel"] == "poisson" and row["track"] == "known/fixed")
known_absolute = next(row for row in gap_rows if row["panel"] == "absolute" and row["track"] == "known/fixed")
failed_calls = sum(row["status"] != "ok" for row in records)

coverage_text = f"{minimum_new_coverage:.4f}–{maximum_new_coverage:.4f}"
absolute_text = f"{absolute_change['width_percent']:.2f} (SE {absolute_change['width_percent_se']:.2f})"
small_width_text = f"{small_change['width_percent']:.2f} (SE {small_change['width_percent_se']:.2f})"
small_mse_text = f"{small_change['point_mse_change']:.3f} (SE {small_change['point_mse_change_se']:.3f})"
known_absolute_text = f"{known_absolute['width_gap_pct']:.2f} (SE {known_absolute['MC_SE_points']:.2f})"
known_poisson_text = f"{known_poisson['width_gap_pct']:.2f} (SE {known_poisson['MC_SE_points']:.2f})"


## Interpretation and limits

All 3,200 method/draw records completed, with {eval}`failed_calls` failed calls. New discovery coverage ranges from {eval}`coverage_text` across the eight panels; the per-panel Monte Carlo SE, including the larger uncertainty at n=80, is shown above. The paired absolute-scale width change is {eval}`absolute_text` percent; SE is in percentage points. Its scale driver is selected in all 50 draws.

The small panel records a paired width increase of {eval}`small_width_text` percent and point-MSE increase {eval}`small_mse_text` squared response units. It selects a mean term in 16/50 draws, compared with 37/50 under the older all-80-row selection. This tradeoff is recorded in [#415](https://github.com/fcela/mboostrs/issues/415); the new workflow uses 40 rows for selection and preserves calibration independence. Its width remains about 1% below retrospective eligible LightGBM split, with substantial uncertainty. That width comparison does not measure the reduction
in mean-term selection from using 40 rather than 80 selection rows.

The fixed supplied references retain width gaps of {eval}`known_absolute_text` percent on absolute Gaussian scale [#630](https://github.com/fcela/mboostrs/issues/630), and {eval}`known_poisson_text` percent on Poisson [#636](https://github.com/fcela/mboostrs/issues/636). Percentage-gap SEs are in percentage points. These references use fixed 100-update fits, not oracle-optimal tuning. The Poisson reference has better point MSE and interval score than its narrower competitor, so the width deficit alone does not diagnose a solver failure; constant-width construction and the count-support definition of width require investigation.

The row-exclusion and response-mutation tests establish the new selection/calibration boundary by construction. The empirical tables test its coverage and costs on the frozen panels. Successful reservation does not establish that the selected formula is correct, that its effect intervals account for selection, or that every interval is efficient. The sharp-mean ambiguity [#629](https://github.com/fcela/mboostrs/issues/629) and remaining varying-scale efficiency investigation [#630](https://github.com/fcela/mboostrs/issues/630) remain relevant.

The earlier 300-row structure-recovery tests now supply additional independently reserved calibration rows to preserve their original selection sample and assertions. Separate tests exercise the default half holdout, reduced sample sizes, missed predictors, budget refusal and invalid calibration responses. No adoption threshold or stopping rule was tuned to recover those fixtures. Calls below the required selection/calibration sizes fail explicitly.

A separate response-unit inconsistency in the existing internal-split LSS path is filed in [#633](https://github.com/fcela/mboostrs/issues/633). The new external path inverse-transforms response quantiles before raw-response calibration; it leaves an unavailable original-scale mean unset rather than calling an inverse-transformed location an expectation. The frozen confirmation panels here have untransformed responses, so that development probe does not select or grade a method in this battery.

Within these IID panels, the new known-specification and discovery tracks support **D9 score 3**: empirical coverage is approximately nominal, with independent-draw uncertainty and failures reported. Extrapolation adaptation is unmeasured, so full score 4 is not established. This study addresses D9 prediction uncertainty within its scope. Source exclusion and exchangeability assumptions establish when the reported marginal bound applies; empirical standard-process coverage is assessed per track. Conditional, grouped and temporal coverage, post-selection effect uncertainty and other scorecard dimensions remain unmeasured. Remaining width gaps and regressions are retained in their issues; calibration independence alone does not establish a 1% efficiency target.

## Reproduction and provenance

The protocol precedes implementation and measurement. Baseline is immutable spread-MR head `105b1b5`; the new implementation was committed as `59b2cf1` before its complete confirmation run. Both use the same release extension and unchanged Rust numerical source. Raw outcomes retain every failure, warning, chosen formula/family/stopping count, input/prediction digest and loaded source identity.

Run `PYTHONPATH=<baseline>/python:. .venv/bin/python -m benches.study_discovery_calibration --phase baseline` and the same command with the new source and `--phase fixed`. The runner checkpoints completed calls and refuses to overwrite its JSON output. Downloaded evidence may be compressed losslessly; the notebook reads gzip directly and does not pool or replace draws. The long competitor fits stay outside the book build; the public example and every table execute in the notebook.

Development failures and their corrections remain available in the [validation record](discovery_calibration/VALIDATION.md). They are distinct from statistical method failures, which stay in the final measurement tables. No failed or unfavorable confirmation draw is replaced.